## GigaAM CTC / RNNT Fine-tuning

This notebook describes the dataset format and provides fine-tuning examples for CTC and RNNT models in end-to-end and raw setups, using activation checkpointing, RNNT loss sub-batching, gradient accumulation, DDP, and encoder freezing for faster, more memory-efficient training.

### Dataset

#### Format

The training pipeline supports a `.tsv` manifest format like the example below.

Each row describes one sample:

* `path` - relative or absolute path to the audio file
* `duration` - audio length in seconds
* `transcription` - reference text for this audio sample

Example:

```tsv
path    duration    transcription
audio/train/000000.wav    5.265    Вот только они совсем не радовали, а, напротив, потрясали и ужасали.
audio/train/000001.wav    4.268    Убедившись, что никого нет, он приблизился ко мне и понизил голос.
```

Usage:

```bash
python train.py \
    --train_manifest /path/to/train/manifest.tsv \
    --val_manifest /path/to/val/manifest.tsv \
    ...

python eval.py \
    --eval_manifest /path/to/eval/manifest.tsv \
    ...
```

**Note:** By default, samples in the training and validation sets are filtered by duration to the `[0.1s, 20s]` range. You can change these limits with `min_duration` and `max_duration`.

#### Loading data

In [1]:
from utils import load_tonebooks

load_tonebooks("data", max_duration=25.0, workers=64);

Loading Vikhrmodels/ToneBooks...


Loading dataset shards:   0%|          | 0/21 [00:00<?, ?it/s]

Splits: ['train', 'validation'], train=91976, val=4841


train (91976):   0%|          | 0/91976 [00:00<?, ?it/s]

val (4841):   0%|          | 0/4841 [00:00<?, ?it/s]

  data/manifest_train.tsv (91472 samples)
  data/manifest_val.tsv (4809 samples)

Done! Manifests at data


### Evaluation

##### Run evaluation for pretrained model

In [1]:
! python -u eval.py \
    --model_name v3_e2e_rnnt \
    --eval_manifest data/manifest_val.tsv \
    --disable_tqdm

filtered by duration: 0/4809 samples (0.0%), 0.00/9.09 h (0.0%)
Loaded 4809 samples
Saved predictions to data/predictions/manifest_val/v3_e2e_rnnt/preds.jsonl
WER e2e: 19.02% (13018/68433 words)
WER raw: 5.01% (3292/65669 words)


##### Check its predictions

In [2]:
import json

with open("data/predictions/manifest_val/v3_e2e_rnnt/preds.jsonl") as file:
    for idx, line in zip(range(3), file):
        d = json.loads(line)
        print(f"Sample {idx}:\nRef:  {d['text']}\nPred: {d['pred_text']}\n")

Sample 0:
Ref:  Сегодня исследователям иногда удается наблюдать эволюцию, что называется, в режиме реального времени.
Pred: Сегодня исследователям иногда удаётся наблюдать эволюцию, что называется, в режиме реального времени.

Sample 1:
Ref:  Много тысячелетий спустя, уже через долгое время после того, как болезнь сама полностью иссякла, независимо друг от друга появились несколько поистине великих людей.
Pred: Много тысячелетий спустя, уже через долгое время после того, как болезнь сама полностью иссякла, независимо друг от друга появились несколько, поистине великих людей.

Sample 2:
Ref:  А Марстен... ну, пожалуй, и Дарвальд по-своему прав.
Pred: А Марстон? Ну, пожалуй, и Дарвальд по-своему прав.



### GigaAM-CTC fine-tuning

#### End-to-end CTC, 2 GPU, ~70Gb VRAM, 4min per epoch

In [1]:
! python -u train.py \
    --train_manifest data/manifest_train.tsv \
    --val_manifest data/manifest_val.tsv \
    --model_name v3_e2e_ctc \
    --max_epochs 3 \
    --val_check_interval 0.5 \
    --batch_size 64 \
    --eval_batch_size 64 \
    --precision bf16 \
    --devices 2 \
    --lr 8e-5 \
    --disable_tqdm

Global seed set to 42
Experiment: v3e2ectc_lr8e-5_wd0.01_b64_2gpu_3ep_vci0.5_dur0.1-20s_pr-bf16
Loading pretrained v3_e2e_ctc ...
Mode: ctc | vocab=256, blank=256
Train: 89214  Val: 4691
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
`Trainer(limit_val_batches=1.0)` was configured so 100% of the batches will be used..
Running initial validation...
[rank: 0] Global seed set to 42
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/2
[rank: 1] Global seed set to 42
Experiment: v3e2ectc_lr8e-5_wd0.01_b64_2gpu_3ep_vci0.5_dur0.1-20s_pr-bf16
Loading pretrained v3_e2e_ctc ...
Mode: ctc | vocab=256, blank=256
Train: 89214  Val: 4691
Running initial validation...
[rank: 1] Global seed set to 42
Initializing distributed: GLOBAL_RANK: 1, MEMBER: 2/2
---------------------------------------------------------------------------------------------------

##### Evaluate model

*Note:* Eval WER is higher, because for the train / val we exclude audio lengths outside the `[0.1s, 20s]` by default.

In [1]:
import glob

ckpts = glob.glob("checkpoints/models/v3e2ectc_*/gigaam-*.ckpt")
ckpt = min(ckpts, key=lambda x: float(x.split("val_wer=")[-1][: -len(".ckpt")]))
print("evaluating:", ckpt)
!python -u eval.py --checkpoint "{ckpt}" --eval_manifest data/manifest_val.tsv --disable_tqdm

filtered by duration: 0/4809 samples (0.0%), 0.00/9.09 h (0.0%)
Loaded 4809 samples
Saved predictions to data/predictions/manifest_val/v3e2ectc_lr8e-5_wd0.01_b64_2gpu_3ep_vci0.5_dur0.1-20s_pr-bf16/step_002088/preds.jsonl
WER e2e: 7.63% (5223/68433 words)
WER raw: 2.45% (1612/65669 words)


In [2]:
import json

step = ckpt.split("step=")[-1].split("-")[0]
preds_path = glob.glob(f"data/predictions/manifest_val/v3e2ectc_*/step_{step}/preds.jsonl")[0]
with open(preds_path) as file:
    for idx, line in zip(range(3), file):
        d = json.loads(line)
        print(f"Sample {idx}:\nRef:  {d['text']}\nPred: {d['pred_text']}\n")

Sample 0:
Ref:  Сегодня исследователям иногда удается наблюдать эволюцию, что называется, в режиме реального времени.
Pred: Сегодня исследователям иногда удается наблюдать эволюцию, что называется, в режиме реального времени.

Sample 1:
Ref:  Много тысячелетий спустя, уже через долгое время после того, как болезнь сама полностью иссякла, независимо друг от друга появились несколько поистине великих людей.
Pred: Много тысячелетий спустя, уже через долгое время после того, как болезнь сама полностью иссякла, независимо друг от друга, появились несколько поистине великих Людей.

Sample 2:
Ref:  А Марстен... ну, пожалуй, и Дарвальд по-своему прав.
Pred: А Марстен. ну, пожалуй, и Дарвальд по-своему прав.



##### The checkpoint can now be used for `load_model` and subsequent workflows

In [1]:
import torch

import gigaam

finetuned_model = gigaam.load_model(ckpt)
finetuned_model.to_onnx("onnx", dtype=torch.float16)

Successfully ported onnx v3_e2e_ctc to onnx/v3_e2e_ctc.onnx.


In [1]:
from gigaam.onnx_utils import load_onnx, infer_onnx
from gigaam.utils import AudioDataset
from utils import compute_wer

sessions, model_cfg = load_onnx("onnx", "v3_e2e_ctc")

texts = infer_onnx(
    "data/manifest_val.tsv", model_cfg, sessions, batch_size=64, progress=False,
)

ds = AudioDataset("data/manifest_val.tsv")
results = [
    {"text": ds.samples[i].text, "pred_text": texts[i]}
    for i in range(len(texts))
]
wer_e2e, wer_raw, e2e_err, e2e_w, raw_err, raw_w = compute_wer(results)

print(
    f"WER e2e: {wer_e2e:.2f}% ({e2e_err}/{e2e_w} words)\n"
    f"WER raw: {wer_raw:.2f}% ({raw_err}/{raw_w} words)"
)

filtered by duration: 0/4809 samples (0.0%), 0.00/9.09 h (0.0%)
filtered by duration: 0/4809 samples (0.0%), 0.00/9.09 h (0.0%)
WER e2e: 7.63% (5219/68433 words)
WER raw: 2.45% (1609/65669 words)


##### Compare with checkpoint before tuning

In [3]:
! python -u eval.py \
    --model_name v3_e2e_ctc \
    --eval_manifest data/manifest_val.tsv \
    --disable_tqdm

filtered by duration: 0/4809 samples (0.0%), 0.00/9.09 h (0.0%)
Loaded 4809 samples
Saved predictions to data/predictions/manifest_val/v3_e2e_ctc/preds.jsonl
WER e2e: 19.34% (13235/68433 words)
WER raw: 5.21% (3423/65669 words)


In [4]:
with open("data/predictions/manifest_val/v3_e2e_ctc/preds.jsonl") as file:
    for idx, line in zip(range(3), file):
        d = json.loads(line)
        print(f"Sample {idx}:\nRef:  {d['text']}\nPred: {d['pred_text']}\n")

Sample 0:
Ref:  Сегодня исследователям иногда удается наблюдать эволюцию, что называется, в режиме реального времени.
Pred: Сегодня исследователям иногда удаётся наблюдать эволюцию, что называется, в режиме реального времени.

Sample 1:
Ref:  Много тысячелетий спустя, уже через долгое время после того, как болезнь сама полностью иссякла, независимо друг от друга появились несколько поистине великих людей.
Pred: Много тысячелетий спустя, уже через долгое время после того, как болезнь сама полностью иссякла, независимо друг от друга появились несколько, поистине великих людей.

Sample 2:
Ref:  А Марстен... ну, пожалуй, и Дарвальд по-своему прав.
Pred: А Марстон? Ну, пожалуй, и Дарвольд по-своему прав.



#### Add activation ckpt, ~12Gb VRAM, 5.5min per epoch

For CTC models, most of the memory is consumed by encoder activations, so using activation checkpointing would be beneficial.

In [1]:
! python -u train.py \
    --train_manifest data/manifest_train.tsv \
    --val_manifest data/manifest_val.tsv \
    --model_name v3_e2e_ctc \
    --max_epochs 3 \
    --val_check_interval 0.5 \
    --batch_size 64 \
    --eval_batch_size 64 \
    --activation_checkpointing \
    --precision bf16 \
    --devices 2 \
    --lr 8e-5 \
    --disable_tqdm

Global seed set to 42
Experiment: v3e2ectc_lr8e-5_wd0.01_b64_2gpu_3ep_vci0.5_acckpt_dur0.1-20s_pr-bf16
Loading pretrained v3_e2e_ctc ...
Encoder: activation checkpointing on (per Conformer layer)
Mode: ctc | vocab=256, blank=256
Train: 89214  Val: 4691
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
`Trainer(limit_val_batches=1.0)` was configured so 100% of the batches will be used..
Running initial validation...
[rank: 0] Global seed set to 42
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/2
[rank: 1] Global seed set to 42
Experiment: v3e2ectc_lr8e-5_wd0.01_b64_2gpu_3ep_vci0.5_acckpt_dur0.1-20s_pr-bf16
Loading pretrained v3_e2e_ctc ...
Encoder: activation checkpointing on (per Conformer layer)
Mode: ctc | vocab=256, blank=256
Train: 89214  Val: 4691
Running initial validation...
[rank: 1] Global seed set to 42
Initializing distribu

#### Gradient accumulation + activation ckpt, 1 GPU, ~9Gb VRAM, 10.5min per epoch

In [5]:
! python -u train.py \
    --train_manifest data/manifest_train.tsv \
    --val_manifest data/manifest_val.tsv \
    --model_name v3_e2e_ctc \
    --max_epochs 3 \
    --val_check_interval 0.5 \
    --batch_size 32 \
    --accumulate_grad_batches 4 \
    --eval_batch_size 64 \
    --activation_checkpointing \
    --precision bf16 \
    --lr 8e-5 \
    --disable_tqdm

Global seed set to 42
Experiment: v3e2ectc_lr8e-5_wd0.01_b32_agb4_3ep_vci0.5_acckpt_dur0.1-20s_pr-bf16
Loading pretrained v3_e2e_ctc ...
Encoder: activation checkpointing on (per Conformer layer)
Mode: ctc | vocab=256, blank=256
Train: 89214  Val: 4691
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
`Trainer(limit_val_batches=1.0)` was configured so 100% of the batches will be used..
Running initial validation...
[rank: 0] Global seed set to 42
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/1
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 1 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
  [val] st

### GigaAM-RNNT fine-tuning

#### Raw RNNT, 2 GPUs, ~65Gb VRAM, 8min per epoch

We reduce the number of evaluation samples to speed up validation, since RNNT inference is slower.

In [6]:
! python train.py \
    --model_name v3_rnnt \
    --train_manifest data/manifest_train.tsv \
    --val_manifest data/manifest_val.tsv \
    --raw_text \
    --max_epochs 2 \
    --val_check_interval 0.5 \
    --batch_size 16 \
    --eval_batch_size 64 \
    --lr 2e-5 \
    --precision bf16 \
    --devices 2 \
    --val_first_batches 50 \
    --disable_tqdm

Global seed set to 42
Experiment: v3rnnt_lr2e-5_wd0.01_b16_2gpu_2ep_vci0.5_vfb50_raw_dur0.1-20s_pr-bf16
Loading pretrained v3_rnnt ...
Mode: rnnt | vocab=33, blank=33
Train: 89003  Val: 4682
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Running initial validation...
[rank: 0] Global seed set to 42
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/2
[rank: 1] Global seed set to 42
Experiment: v3rnnt_lr2e-5_wd0.01_b16_2gpu_2ep_vci0.5_vfb50_raw_dur0.1-20s_pr-bf16
Loading pretrained v3_rnnt ...
Mode: rnnt | vocab=33, blank=33
Train: 89003  Val: 4682
Running initial validation...
[rank: 1] Global seed set to 42
Initializing distributed: GLOBAL_RANK: 1, MEMBER: 2/2
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with

#### RNNT loss sub-batch, ~22Gb VRAM, 9min per epoch

Unlike CTC, RNNT uses a lot of memory for spikes during RNNT loss computation with `[batch, audio_seq_len, text_seq_len + 1, vocab_size]` tensor; a simple way to reduce memory usage is to use micro-batching and compute the loss separately.

In [1]:
! python train.py \
    --model_name v3_rnnt \
    --train_manifest data/manifest_train.tsv \
    --val_manifest data/manifest_val.tsv \
    --raw_text \
    --max_epochs 2 \
    --val_check_interval 0.5 \
    --batch_size 16 \
    --rnnt_subbatch_size 2 \
    --eval_batch_size 64 \
    --lr 2e-5 \
    --precision bf16 \
    --devices 2 \
    --val_first_batches 50 \
    --disable_tqdm

Global seed set to 42
Experiment: v3rnnt_lr2e-5_wd0.01_b16_2gpu_2ep_vci0.5_vfb50_raw_dur0.1-20s_pr-bf16
Loading pretrained v3_rnnt ...
Mode: rnnt | vocab=33, blank=33
Train: 89003  Val: 4682
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Running initial validation...
[rank: 0] Global seed set to 42
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/2
[rank: 1] Global seed set to 42
Experiment: v3rnnt_lr2e-5_wd0.01_b16_2gpu_2ep_vci0.5_vfb50_raw_dur0.1-20s_pr-bf16
Loading pretrained v3_rnnt ...
Mode: rnnt | vocab=33, blank=33
Train: 89003  Val: 4682
Running initial validation...
[rank: 1] Global seed set to 42
Initializing distributed: GLOBAL_RANK: 1, MEMBER: 2/2
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with

#### Sub-batch + freeze encoder, ~8Gb VRAM, 6min per epoch

For RNNT, it is also possible to freeze the encoder and train only the decoder and joint modules.

In [3]:
! python train.py \
    --model_name v3_rnnt \
    --train_manifest data/manifest_train.tsv \
    --val_manifest data/manifest_val.tsv \
    --raw_text \
    --max_epochs 2 \
    --val_check_interval 0.5 \
    --batch_size 16 \
    --rnnt_subbatch_size 2 \
    --freeze_encoder_epochs -1 \
    --eval_batch_size 64 \
    --lr 2e-4 \
    --precision bf16 \
    --devices 2 \
    --val_first_batches 50 \
    --disable_tqdm

Global seed set to 42
Experiment: v3rnnt_lr0.0002_wd0.01_b16_2gpu_2ep_vci0.5_frenc_vfb50_raw_dur0.1-20s_pr-bf16
Loading pretrained v3_rnnt ...
Mode: rnnt | vocab=33, blank=33
Train: 89003  Val: 4682
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Running initial validation...
[rank: 0] Global seed set to 42
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/2
[rank: 1] Global seed set to 42
Experiment: v3rnnt_lr0.0002_wd0.01_b16_2gpu_2ep_vci0.5_frenc_vfb50_raw_dur0.1-20s_pr-bf16
Loading pretrained v3_rnnt ...
Mode: rnnt | vocab=33, blank=33
Train: 89003  Val: 4682
Running initial validation...
[rank: 1] Global seed set to 42
Initializing distributed: GLOBAL_RANK: 1, MEMBER: 2/2
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registere

## New language from an SSL backbone

The `multilingual_ssl` backbone ships only the preprocessor and the Conformer encoder — there is no ASR head and no vocabulary. Passing it as `--model_name` makes `train.py` attach a randomly initialized character-wise head (CTC or RNN-T) sized to the target alphabet and fine-tune it. Below we adapt it to Armenian using FLEURS.


### Load the dataset

`load_fleurs` downloads a FLEURS language subset (config `<lang>_<region>`, e.g. `hy_am` for Armenian) and writes `manifest_train.tsv` / `manifest_val.tsv`.


In [1]:
from utils import load_fleurs

load_fleurs("data_hy", lang="hy_am", workers=64)

Loading google/fleurs [hy_am]...


train (3053):   0%|          | 0/3053 [00:00<?, ?it/s]

  data_hy/manifest_train.tsv (3050 samples)


val (395):   0%|          | 0/395 [00:00<?, ?it/s]

  data_hy/manifest_val.tsv (394 samples)

Done! Manifests at data_hy


PosixPath('data_hy')

### Fine-tune a CTC head

The alphabet is derived from the manifest transcriptions (`--build_vocab_from_manifest`) and saved for reuse. The head is trained from scratch, so use a higher LR (`3e-4`) and more epochs than for same-language fine-tuning.


In [2]:
! python -u train.py \
    --model_name multilingual_ssl \
    --head ctc \
    --raw_text \
    --train_manifest data_hy/manifest_train.tsv \
    --val_manifest data_hy/manifest_val.tsv \
    --build_vocab_from_manifest \
    --save_vocab data_hy/vocab.json \
    --output_dir ./checkpoints_hy \
    --exp_name hy_ssl_ctc \
    --batch_size 16 \
    --eval_batch_size 32 \
    --lr 3e-4 \
    --weight_decay 1e-2 \
    --warmup_ratio 0.1 \
    --freeze_encoder_epochs 3 \
    --max_epochs 30 \
    --val_check_interval 1.0 \
    --time_masks 10 \
    --save_top_k 1 \
    --activation_checkpointing \
    --disable_tqdm

Seed set to 42
Experiment: hy_ssl_ctc
Loading pretrained multilingual_ssl ...


Saved vocabulary (78 chars) to data_hy/vocab.json
SSL backbone 'multilingual_ssl': building a randomly initialized CTC head (vocab=78 chars + blank), labels: raw (normalized)


Encoder: activation checkpointing on (per Conformer layer)
Mode: ctc | vocab=78, blank=78


filtered by duration: 117/3050 samples (3.8%), 0.75/10.34 h (7.2%)


filtered by duration: 10/394 samples (2.5%), 0.06/1.20 h (5.2%)
Train: 2933  Val: 384


...

  [val] step=5124 epoch=27  val/loss=28.635607  val/wer=0.1533
[epoch 27] time: 37.72 sec


  [val] step=5307 epoch=28  val/loss=28.610809  val/wer=0.1518
[epoch 28] time: 37.56 sec


  [val] step=5490 epoch=29  val/loss=28.588888  val/wer=0.1517
[epoch 29] time: 37.62 sec


`Trainer.fit` stopped: `max_epochs=30` reached.


Best: ./checkpoints_hy/models/hy_ssl_ctc/gigaam-multilingual_ssl-epoch=29-step=005490-val_wer=0.1517.ckpt


### Fine-tune an RNN-T head

Same backbone and vocabulary, with `--head rnnt`. RNN-T head sizes default to the standard GigaAM values (`--rnnt_pred_hidden`, `--rnnt_joint_hidden`, `--rnnt_pred_rnn_layers` to override). We reuse the vocabulary saved above via `--vocab`.


In [3]:
! python -u train.py \
    --model_name multilingual_ssl \
    --head rnnt \
    --raw_text \
    --train_manifest data_hy/manifest_train.tsv \
    --val_manifest data_hy/manifest_val.tsv \
    --vocab data_hy/vocab.json \
    --output_dir ./checkpoints_hy \
    --exp_name hy_ssl_rnnt \
    --batch_size 16 \
    --rnnt_subbatch_size 2 \
    --eval_batch_size 32 \
    --lr 3e-4 \
    --weight_decay 1e-2 \
    --warmup_ratio 0.1 \
    --freeze_encoder_epochs 3 \
    --max_epochs 30 \
    --val_check_interval 1.0 \
    --time_masks 10 \
    --save_top_k 1 \
    --activation_checkpointing \
    --disable_tqdm

Seed set to 42


Experiment: hy_ssl_rnnt
Loading pretrained multilingual_ssl ...


SSL backbone 'multilingual_ssl': building a randomly initialized RNNT head (vocab=78 chars + blank), labels: raw (normalized)


Encoder: activation checkpointing on (per Conformer layer)
Mode: rnnt | vocab=78, blank=78


filtered by duration: 117/3050 samples (3.8%), 0.75/10.34 h (7.2%)


...

  [val] step=4941 epoch=26  val/loss=18.835085  val/wer=0.1496
[epoch 26] time: 59.40 sec


  [val] step=5124 epoch=27  val/loss=18.869791  val/wer=0.1466
[epoch 27] time: 58.64 sec


  [val] step=5307 epoch=28  val/loss=18.851074  val/wer=0.1459
[epoch 28] time: 59.91 sec


  [val] step=5490 epoch=29  val/loss=18.845306  val/wer=0.1459
[epoch 29] time: 58.38 sec
`Trainer.fit` stopped: `max_epochs=30` reached.


Best: ./checkpoints_hy/models/hy_ssl_rnnt/gigaam-multilingual_ssl-epoch=28-step=005307-val_wer=0.1459.ckpt


### Evaluate and load the fine-tuned model

In [4]:
import glob

ckpts = glob.glob("checkpoints_hy/models/hy_ssl_ctc/*.ckpt")
ckpt = min(ckpts, key=lambda x: float(x.split("val_wer=")[-1][:-len(".ckpt")]))
print("evaluating:", ckpt)
!python -u eval.py --checkpoint "{ckpt}" --eval_manifest data_hy/manifest_val.tsv --batch_size 32 --disable_tqdm

evaluating: checkpoints_hy/models/hy_ssl_ctc/gigaam-multilingual_ssl-epoch=29-step=005490-val_wer=0.1517.ckpt


filtered by duration: 0/394 samples (0.0%), 0.00/1.20 h (0.0%)
Loaded 394 samples


Saved predictions to data_hy/predictions/manifest_val/hy_ssl_ctc/step_005490/preds.jsonl
WER e2e: 16.85% (1186/7037 words)
WER raw: 15.61% (1108/7100 words)


In [5]:
import glob

import gigaam

ckpt = sorted(glob.glob("checkpoints_hy/models/hy_ssl_ctc/gigaam-*.ckpt"))[-1]
model = gigaam.load_model(ckpt)
print(model.transcribe("data_hy/audio/val/000000.wav"))

ինչպես տեղեկացնում են մեծ թափ առած թարանը շարունակվել է գիշերը քանի որ իրավապահները չէին հսկում բիշկեկի փողոցները
